<a href="https://colab.research.google.com/github/PovSobek/Manipuladores/blob/main/03_Cinematica_inversa.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# <font color='steelblue'> Introducción a la Cinemática Inversa y Espacio de Trabajo en Robótica </font>

**Material desarrollado por Vicente Esteve-Sala**

![](https://drive.google.com/thumbnail?id=1GrzpXe8zsvQwyY7zTw87VCG4c14sHTvt&sz=w800)

**Fecha última edición**: 13/10/2025

**Licencia**:
<small> © 2025 by <a href="https://cvnet.cpd.ua.es/curriculum-breve/es/esteve-sala-vicente-manuel/283903">Vicente Esteve-Sala</a> is licensed under <a href="https://creativecommons.org/licenses/by-nc-sa/4.0/">CC BY-NC-SA 4.0       </a><small><a rel="license" href="http://creativecommons.org/licenses/by-nc-sa/4.0/"><img alt="Creative Commons License" style="border-width:0" src="https://i.creativecommons.org/l/by-nc-sa/4.0/88x31.png" /></a><br /></small>

Al usar estos contenidos, aceptas los términos de uso, propiedad intelectual y la política de privacidad de la UA.

# Introducción a la Cinemática Inversa y Espacio de Trabajo ⚙️

En el cuaderno anterior, resolvimos la **Cinemática Directa**: a partir de los ángulos de las articulaciones $(\theta_1, \theta_2)$, calculamos la posición del efector final $(x, y)$.

Ahora, abordaremos el problema inverso y más común en la práctica: la **Cinemática Inversa**.
* **Objetivo:** Dada una posición deseada $(x_{obj}, y_{obj})$, ¿qué ángulos $(\theta_1, \theta_2)$ deben adoptar las articulaciones para alcanzarla?

Este problema es más complejo porque:
1.  **No siempre hay solución:** El punto puede estar fuera del alcance del brazo. El área que el brazo sí puede alcanzar se llama **espacio de trabajo**.
2.  **Puede haber múltiples soluciones:** A menudo, el brazo puede alcanzar el mismo punto con dos configuraciones distintas (por ejemplo, "codo arriba" o "codo abajo").

Vamos a explorar estos conceptos.

In [ ]:
# Importamos las bibliotecas que ya conocemos
import numpy as np
import matplotlib.pyplot as plt

# --- ENTRADA DE DATOS INTERACTIVA ---
try:
    print("--- Introduce los PARÁMETROS del ROBOT ---")

    # Le pedimos al usuario que introduzca la longitud del primer eslabón
    L1 = float(input("Introduce la longitud del primer eslabón (L1) en metros: "))

    # Le pedimos al usuario que introduzca la longitud del segundo eslabón
    L2 = float(input("Introduce la longitud del segundo eslabón (L2) en metros: "))

    # --- NUEVO: Solicitar el punto objetivo ---
    print("\n--- Introduce el PUNTO OBJETIVO 🎯 ---")
    x_objetivo = float(input("Introduce la coordenada X del objetivo: "))
    y_objetivo = float(input("Introduce la coordenada Y del objetivo: "))

    # Guardamos las coordenadas en una única variable para mayor comodidad
    punto_objetivo = (x_objetivo, y_objetivo)


    print("\n¡Parámetros definidos con éxito!")
    print(f"Longitudes establecidas: L1 = {L1} m, L2 = {L2} m")
    print(f"Punto objetivo establecido: {punto_objetivo}")

except ValueError:
    print("\nError: Por favor, introduce solo valores numéricos. Vuelve a ejecutar la celda.")

## 1. El Espacio de Trabajo (Workspace)

Antes de intentar calcular los ángulos, debemos saber si el punto es alcanzable. El espacio de trabajo de nuestro robot de 2-DOF es un "anillo".

* **Alcance máximo:** La distancia más lejana que puede alcanzar es $L_1 + L_2$.
* **Alcance mínimo:** La distancia más cercana al origen que puede alcanzar (sin chocar consigo mismo) es $|L_1 - L_2|$.

En la siguiente celda, visualizaremos esta zona. Cualquier punto dentro del área azul es teóricamente alcanzable.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Circle


# --- Visualización del Espacio de Trabajo ---
try:
    alcance_max = L1 + L2
    alcance_min = np.abs(L1 - L2)

    # Creamos la figura y los círculos que definen los límites
    fig, ax = plt.subplots(figsize=(8, 8)) # Creamos una figura y un eje para mayor control

    circulo_externo = Circle((0, 0), alcance_max, color='lightblue', alpha=0.5, label='Espacio de Trabajo (Alcance Máx)')
    circulo_interno = Circle((0, 0), alcance_min, color='white', alpha=0.8, label='Alcance Mín (Singularidad)')

    # Configuramos el gráfico
    ax.add_patch(circulo_externo)
    ax.add_patch(circulo_interno)

    # Dibujar el punto objetivo ---
    x_obj, y_obj = punto_objetivo # Desempaquetamos las coordenadas del punto objetivo

    # Dibujamos el punto objetivo como un círculo rojo
    ax.plot(x_obj, y_obj, 'ro', markersize=8, label=f'Punto Objetivo ({x_obj:.2f}, {y_obj:.2f})')

    plt.title("Espacio de Trabajo del Robot 2-DOF con Punto Objetivo")
    plt.xlabel("Coordenada X (m)")
    plt.ylabel("Coordenada Y (m)")

    # Ajustar los límites del gráfico dinámicamente
    margen = 0.2
    max_coord_display = max(alcance_max, abs(x_obj), abs(y_obj)) + margen # Ajuste para incluir el objetivo si está fuera del alcance pero cerca
    ax.set_xlim(-max_coord_display, max_coord_display)
    ax.set_ylim(-max_coord_display, max_coord_display)

    ax.set_aspect('equal') # Para que los círculos se vean como círculos, no elipses
    plt.grid(True)
    plt.legend() # Mostramos la leyenda para incluir la etiqueta del punto objetivo
    plt.show()

except NameError:
    print("\nError: Asegúrate de haber ejecutado las celdas anteriores que definen L1, L2 y 'punto_objetivo'.")
except Exception as e:
    print(f"\nOcurrió un error al generar la visualización: {e}")

In [ ]:
# Generar dos puntos: uno alcanzable y uno que no lo es
# Usamos los valores de L1 y L2 definidos anteriormente

# # Punto alcanzable (dentro del espacio de trabajo)
# punto_alcanzable = (L1 + L2 - 0.5, 0.5) # Just inside the max reach

# # Punto no alcanzable (fuera del espacio de trabajo)
# punto_no_alcanzable = (L1 + L2 + 1.0, 0.0) # Outside the max reach

# # Probar la función calcular_ik con los puntos
# print("--- Probando con punto alcanzable ---")
# angulos_alcanzable = calcular_ik(punto_alcanzable[0], punto_alcanzable[1], L1, L2)
# if angulos_alcanzable:
#     print(f"Para alcanzar {punto_alcanzable}, los ángulos deben ser:")
#     print(f"  Theta1 = {angulos_alcanzable[0]:.2f} grados")
#     print(f"  Theta2 = {angulos_alcanzable[1]:.2f} grados")

# print("\n--- Probando con punto no alcanzable ---")
# angulos_no_alcanzable = calcular_ik(punto_no_alcanzable[0], punto_no_alcanzable[1], L1, L2)
# if angulos_no_alcanzable:
#     print(f"Para alcanzar {punto_no_alcanzable}, los ángulos deben ser:")
#     print(f"  Theta1 = {angulos_no_alcanzable[0]:.2f} grados")
#     print(f"  Theta2 = {angulos_no_alcanzable[1]:.2f} grados")

Now, let's visualize these points on the workspace plot.

In [ ]:
# import numpy as np
# import matplotlib.pyplot as plt
# from matplotlib.patches import Circle

# # Asegúrate de que L1, L2, punto_alcanzable y punto_no_alcanzable estén definidos

# alcance_max = L1 + L2
# alcance_min = np.abs(L1 - L2)

# fig, ax = plt.subplots(figsize=(8, 8))

# circulo_externo = Circle((0, 0), alcance_max, color='lightblue', alpha=0.5, label='Espacio de Trabajo (Alcance Máx)')
# circulo_interno = Circle((0, 0), alcance_min, color='white', alpha=0.8, label='Alcance Mín (Singularidad)')

# ax.add_patch(circulo_externo)
# ax.add_patch(circulo_interno)

# # Dibujar el punto alcanzable
# ax.plot(punto_alcanzable[0], punto_alcanzable[1], 'go', markersize=8, label=f'Punto Alcanzable ({punto_alcanzable[0]:.2f}, {punto_alcanzable[1]:.2f})')

# # Dibujar el punto no alcanzable
# ax.plot(punto_no_alcanzable[0], punto_no_alcanzable[1], 'ro', markersize=8, label=f'Punto No Alcanzable ({punto_no_alcanzable[0]:.2f}, {punto_no_alcanzable[1]:.2f})')


# plt.title("Espacio de Trabajo del Robot 2-DOF con Puntos de Prueba")
# plt.xlabel("Coordenada X (m)")
# plt.ylabel("Coordenada Y (m)")

# margen = 0.5 # Aumentamos el margen para asegurar que los puntos fuera del alcance se vean
# max_coord_display = max(alcance_max, abs(punto_alcanzable[0]), abs(punto_alcanzable[1]), abs(punto_no_alcanzable[0]), abs(punto_no_alcanzable[1])) + margen
# ax.set_xlim(-max_coord_display, max_coord_display)
# ax.set_ylim(-max_coord_display, max_coord_display)

# ax.set_aspect('equal')
# plt.grid(True)
# plt.legend()
# plt.show()

## 2. La Matemática de la Cinemática Inversa (Solución Geométrica)

Para encontrar $\theta_1$ y $\theta_2$ a partir de $(x, y)$, podemos usar geometría y el **Teorema del Coseno**. Observa el triángulo formado por los dos eslabones y la línea imaginaria desde el origen al punto $(x, y)$.



Las ecuaciones para la solución "codo arriba" son:
<br><br>
$$ D = \sqrt{x^2 + y^2} $$
<br><br>
$$ \theta_2 = \arccos\left(\frac{D^2 - L_1^2 - L_2^2}{2 L_1 L_2}\right) $$
<br><br>
$$ \theta_1 = \text{atan2}(y, x) - \arctan\left(\frac{L_2 \sin(\theta_2)}{L_1 + L_2 \cos(\theta_2)}\right) $$
<br><br>

**Nota:** La función `atan2(y, x)` es una versión especial del arco-tangente que devuelve el ángulo correcto en los cuatro cuadrantes.
<br><br>
### **Ejercicio Práctico:**



In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Circle

# --------------------------------------------------------------------

def calcular_ik(x, y, L1, L2):
    """
    Calcula los ángulos de las articulaciones para alcanzar un punto (x, y).
    Devuelve (theta1_grados, theta2_grados) o None si no es alcanzable.
    """
    distancia = np.sqrt(x**2 + y**2)
    alcance_max = L1 + L2
    alcance_min = np.abs(L1 - L2)

    # 1. Comprobar si el punto está dentro del espacio de trabajo.
    # Si la distancia está fuera del rango [alcance_min, alcance_max], no es alcanzable.
    if not (alcance_min <= distancia <= alcance_max):
        print(f"Error: El punto ({x:.2f}, {y:.2f}) está fuera del espacio de trabajo.")
        return None

# ▼▼▼ TU CÓDIGO AQUÍ ▼▼▼

    # 2. Calcular theta2 (en radianes) usando el Teorema del Coseno.
    # Se añade np.clip para evitar errores de dominio por imprecisiones numéricas.
    cos_arg_th2 = np.clip((distancia**2-L1**2-L2**2)/(2*L1*L2)) # Reemplaza el 0 con la fórmula correcta
    theta2_rad = np.arccos(cos_arg_th2) # Reemplaza el 0 con la fórmula correcta

    # 3. Calcular theta1 (en radianes).
    beta = np.arctan(y/x) # Reemplaza el 0 con la fórmula correcta
    alpha = np.arctan(L2*np.sin(theta2_rad)/(L1+L2*np.cos(theta2_rad))) # Reemplaza el 0 con la fórmula correcta
    theta1_rad = beta - alpha # Reemplaza el 0 con la fórmula correcta

# ▲▲▲ FIN DE TU CODIGO ▲▲▲


    # Convertimos los resultados a grados antes de devolverlos.
    theta1_grados = np.rad2deg(theta1_rad)
    theta2_grados = np.rad2deg(theta2_rad)

    return (theta1_grados, theta2_grados)

# --- Realizamos los cálculos con la función ---
angulos = calcular_ik(punto_objetivo[0], punto_objetivo[1], L1, L2)

if angulos:
    print(f"\nPara alcanzar el punto {punto_objetivo}, los ángulos calculados son:")
    print(f"  Theta1 = {angulos[0]:.2f} grados")
    print(f"  Theta2 = {angulos[1]:.2f} grados")

# --- Dibujamos el espacio de trabajo y el punto objetivo ---
alcance_max_viz = L1 + L2
alcance_min_viz = np.abs(L1 - L2)
fig, ax = plt.subplots(figsize=(8, 8))

circulo_externo = Circle((0, 0), alcance_max_viz, color='lightblue', alpha=0.5, label='Espacio de Trabajo')
circulo_interno = Circle((0, 0), alcance_min_viz, color='white')
ax.add_patch(circulo_externo)
ax.add_patch(circulo_interno)

# Dibujamos el punto objetivo
x_obj, y_obj = punto_objetivo
ax.plot(x_obj, y_obj, 'ro', markersize=10, label=f'Punto Objetivo ({x_obj:.2f}, {y_obj:.2f})')

# Configuramos el gráfico
plt.title("Espacio de Trabajo y Punto Objetivo")
plt.xlabel("Coordenada X (m)")
plt.ylabel("Coordenada Y (m)")
margen = 0.3
limite_grafico = max(alcance_max_viz, abs(x_obj), abs(y_obj)) + margen
ax.set_xlim(-limite_grafico, limite_grafico)
ax.set_ylim(-limite_grafico, limite_grafico)
ax.set_aspect('equal')
plt.grid(True)
plt.legend()
plt.show()

**TAREA**

Completa la función `calcular_ik` en la celda anterior. La función debe:
1.  Recibir una coordenada objetivo `(x, y)` y las longitudes `L1`, `L2`.
2.  **Primero**, comprobar si el punto está dentro del espacio de trabajo. Si no lo está, debe devolver `None`.
3.  **Si es alcanzable**, debe calcular `theta1` y `theta2` (en grados) usando las fórmulas de arriba y devolverlos.
4. **Genera dos posiciones diferentes, una alcanzable y otra que no**
5. **Realiza capturas de pantalla de las dos posiciones**
6. **Entrega en un pdf las capturas de pantallas y el código generado**
